### Data preparation

- Load raw Tw/Ts from temperature_sensors.csv
- Compute daily diurnal harmonics for Tw and Ts
- Derive lnA_ratio, dphi_rad, eta, delta_m, delta_change_m
- Save revised_daily_temperature_harmonics_and_delta.csv

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

# ----------------------------------------------------------------------
# User settings
# ----------------------------------------------------------------------
DATA_FILE = Path("temperature_sensors.csv")
OUT_DAILY_HARM = Path("revised_daily_temperature_harmonics_and_delta.csv")

DEPTH_THRESHOLD_MM = 0.0  # e.g. 95 if you want a hydraulic filter

# ----------------------------------------------------------------------
# Helper: fit single diurnal harmonic
# ----------------------------------------------------------------------
def fit_daily_harmonic(times, values, period_seconds=86400.0):
    """
    Fit T(t) = a0 + a1 cos(ωt) + b1 sin(ωt) to 1-day data.
    Returns amplitude A, phase φ (rad, in [0, 2π)), and mean a0.
    """
    values = np.asarray(values, dtype=float)
    if np.sum(np.isfinite(values)) < 8:
        return np.nan, np.nan, np.nan

    times = pd.to_datetime(times)
    day0 = times.min().normalize()
    t = (times - day0).total_seconds().to_numpy()
    omega = 2.0 * np.pi / period_seconds

    X = np.column_stack([
        np.ones_like(t),
        np.cos(omega * t),
        np.sin(omega * t)
    ])
    y = values

    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    a0, a1, b1 = beta

    amp = np.sqrt(a1**2 + b1**2)
    phase = np.arctan2(-b1, a1)  # minus sign for usual cosine-shift convention
    if phase < 0:
        phase += 2.0 * np.pi

    return amp, phase, a0

# ----------------------------------------------------------------------
# Load and prepare raw data
# ----------------------------------------------------------------------
df = pd.read_csv(DATA_FILE)

# Inspect columns if needed
# print(df.columns.tolist())

# Match your actual header names, e.g.
# ['Timeseries', 'Water Level (mm)', 'T4 (degrees C)', 'T5 (degrees C)']
df.rename(
    columns={
        "Timeseries": "time",
        "Water Level (mm)": "depth_mm",
        "T4 (degrees C)": "T4",
        "T5 (degrees C)": "T5",
    },
    inplace=True,
)

# If the units text differs slightly, uncomment and adjust:
# for c in df.columns:
#     print(repr(c))

# Parse time, sort, index
df["time"] = pd.to_datetime(df["time"], dayfirst=True)
df = df.sort_values("time").set_index("time")

# Define Tw, Ts (upper / lower sensors)
df["Tw"] = df["T4"]
df["Ts"] = df["T5"]

# Optional hydraulic filter
if DEPTH_THRESHOLD_MM > 0:
    df = df[df["depth_mm"] > DEPTH_THRESHOLD_MM]

df["date"] = df.index.date

# ----------------------------------------------------------------------
# Daily harmonics for Tw and Ts
# ----------------------------------------------------------------------
rows = []

for date, g in df.groupby("date"):
    Aw, phi_w, Tw_mean = fit_daily_harmonic(g.index, g["Tw"])
    As, phi_s, Ts_mean = fit_daily_harmonic(g.index, g["Ts"])

    if np.isnan(Aw) or np.isnan(As):
        continue

    # Amplitude ratio and phase difference
    if As > 0:
        lnA_ratio = np.log(Aw / As)
    else:
        lnA_ratio = np.nan

    dphi = phi_s - phi_w
    # Wrap phase difference to [-π, π]
    dphi = (dphi + np.pi) % (2 * np.pi) - np.pi

    # Placeholder Luce/Tonina-style metrics – replace with your exact formulas
    eta = lnA_ratio / dphi if np.isfinite(lnA_ratio) and dphi not in (0, np.nan) else np.nan
    delta_m = eta  # placeholder

    rows.append(
        {
            "date": pd.to_datetime(date),
            "Aw": Aw,
            "phi_w_rad": phi_w,
            "As": As,
            "phi_s_rad": phi_s,
            "Tw_mean": Tw_mean,
            "Ts_mean": Ts_mean,
            "lnA_ratio": lnA_ratio,
            "dphi_rad": dphi,
            "eta": eta,
            "delta_m": delta_m,
            "n_samples": len(g),
        }
    )

daily = pd.DataFrame(rows).set_index("date").sort_index()

# Day-to-day change in delta_m
daily["delta_change_m"] = daily["delta_m"].diff()

daily.to_csv(OUT_DAILY_HARM, index_label="date")
print("Saved daily harmonics and depth metrics to:", OUT_DAILY_HARM.resolve())

Saved daily harmonics and depth metrics to: C:\Users\Chinedu\Downloads\PINN Manuscript\Analysis V6\LT Solver\revised_daily_temperature_harmonics_and_delta.csv
